In [2]:
import os
print(os.getcwd())

c:\Users\josen\Desktop\pricing-wayback-scraper\notebooks


In [3]:
import pandas as pd

df = pd.read_csv("../data/news_domain_cleaned_sim.csv")
print(df.shape)
df.describe(include="all")

(299, 5)


,domain,top_domain,traffic,primary_country_code,primary_country_share
count,299,299,2.990000e+02,299,299.000000
unique,299,299,NaN,44,NaN
top,news.google.com,google.com,NaN,US,NaN
freq,1,1,NaN,71,NaN
mean,NaN,NaN,4.659310e+08,NaN,0.768062
std,NaN,NaN,5.495275e+09,NaN,0.238369
min,NaN,NaN,3.546611e+07,NaN,0.092819
25%,NaN,NaN,4.660485e+07,NaN,0.701439
50%,NaN,NaN,6.427469e+07,NaN,0.857779
75%,NaN,NaN,9.642194e+07,NaN,0.939717


In [ ]:
df_sorted = df.sort_values(by="traffic", ascending=False) #First rank domains by popularity (more popular = first)
df_sorted.head(20)

,domain,top_domain,traffic,primary_country_code,primary_country_share
0,news.google.com,google.com,94881645895,US,0.192977
1,arz.m.wikipedia.org,wikipedia.org,5143703369,US,0.222355
2,au.sports.yahoo.com,yahoo.com,4495242317,US,0.487774
3,article.yahoo.co.jp,yahoo.co.jp,1846124587,JP,0.982874
4,weather.com,weather.com,1644982011,US,0.417399
5,social.technet.microsoft.com,microsoft.com,992365009,US,0.171637
6,sway.office.com,office.com,873983152,US,0.297375
7,finance.naver.com,naver.com,845945497,KR,0.916839
8,arabic.cnn.com,cnn.com,674632188,US,0.744267
9,myaccount.nytimes.com,nytimes.com,669840885,US,0.685587


In [ ]:
import requests #using requests library to access response status codes to help speed up process of checking for paywalls

response = requests.get("https://example.com/subscribe")
print(response.status_code)

404


In [26]:
import requests

response = requests.get("https://myaccount.nytimes.com")
print(response.status_code)
print(response.url)
print(response.history)

403
https://myaccount.nytimes.com/auth/login?response_type=cookie&client_id=acct&redirect_uri=https%3A%2F%2Fmyaccount.nytimes.com%2F
[<Response [302]>]


In [42]:
import requests

# Checks a domain for common subscription-related URL paths and classifies
# each response as a positive, negative, or needs-manual-review signal.

def check_domains(domain, paths=["subscribe", "membership", "join", "pricing"]): 
    results = []
    for path in paths:
        url = f"https://{domain}/{path}" 
        try:
            response = requests.get(url, timeout=5)
            if response.status_code == 200:
                # Exact match: final URL path is identical to what we requested
                # (no redirect elsewhere). Strong signal the page is real.
                if response.url.endswith(f"https://{domain}/{path}"): 
                    verdict = "positive"
                elif any(keyword in response.url for keyword in ["subscri", "member", "plan"]):
                    verdict = "likely positive"
                else:
                    # 200, but the final URL differs from what we requested —
                    # could be a redirect to a real subscribe page (like NYT's
                    # /subscription redirect) OR a bounce to the bare homepage
                    # (like Naver). Can't tell which from status code alone,
                    # so flag for a human to glance at final_url.
                    verdict = "manual positive" 
            elif response.status_code == 404:
                # No page at this path = confident negative, no review needed.
                verdict = "clean negative" 
            else:
                # Any other status code (500, 403, etc.) = behavior unclear,
                # needs manual human look.
                verdict = "manual negative"
            results.append({
    "domain": domain,
    "path": path,
    "status": response.status_code,
    "final_url": response.url,
    "verdict": verdict
})
        except requests.exceptions.RequestException as e:
            # Request itself failed (timeout, DNS error, connection refused)
            # this is the "could not be crawled" case from the task doc,
            # distinct from "no subscription."
            results.append({
        "domain": domain,
        "path": path,
        "status": None,
        "final_url": None,
        "verdict": "error",
        "error": str(e)
        })
    return results


In [43]:
check_domains("www.nytimes.com")

[{'domain': 'www.nytimes.com',
  'path': 'subscribe',
  'status': 200,
  'final_url': 'https://www.nytimes.com/subscription?campaignId=9FRJJ',
  'verdict': 'likely positive'},
 {'domain': 'www.nytimes.com',
  'path': 'membership',
  'status': 404,
  'final_url': 'https://www.nytimes.com/membership',
  'verdict': 'clean negative'},
 {'domain': 'www.nytimes.com',
  'path': 'join',
  'status': 404,
  'final_url': 'https://www.nytimes.com/join',
  'verdict': 'clean negative'},
 {'domain': 'www.nytimes.com',
  'path': 'pricing',
  'status': 404,
  'final_url': 'https://www.nytimes.com/pricing',
  'verdict': 'clean negative'}]